<a href="https://colab.research.google.com/github/sarahzeyad19/asphalt1/blob/claude%2Frut-20k-modeling-workflow-23873s/Rut20k_Reorganized_Colab_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUT_20K Machine Learning Regression Workflow — Clean Google Colab Version

This notebook reorganizes the original single Python workflow into clear, step-by-step research cells.  
The workflow is designed for asphalt rutting prediction (`Rut_20k`) using routine JMF variables and true RBR features.

**Main rules used in this notebook**

- The locked test set is split first and protected until the final evaluation cell.
- `MixDesignKey` is used as the grouping variable so the same mix design does not leak across train, validation, and test.
- All preprocessing is inside `Pipeline` / `ColumnTransformer` objects and is fit only on the training data during model selection.
- Validation is used for model selection; the locked test is used once only after the final model is selected.
- Outliers are screened and interpreted, not automatically removed.

## Optional Colab Setup

Run this cell only if the packages are missing in your Colab environment.  
For a fast first run, keep `QUICK_RUN = True` in the settings cell. For the paper/final run, set `QUICK_RUN = False` and increase the tuning budgets.

In [ ]:
# Optional installs for Google Colab.
# Uncomment only the packages you need.

# !pip install -q xgboost lightgbm catboost shap optuna openpyxl
# !pip install -q tabpfn
# !pip install -q autogluon.tabular

print("Setup cell loaded. Uncomment installation lines only when needed.")

# Step 1: Data Audit

## 1.1 Load Libraries and Data

This cell imports the libraries, defines project settings, creates output folders, and sets the target, grouping variable, and split proportions.

In [ ]:
import json
import math
import os
import re
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    IsolationForest,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    XGBRegressor = None
    HAS_XGBOOST = False

try:
    from lightgbm import LGBMRegressor
    HAS_LIGHTGBM = True
except Exception:
    LGBMRegressor = None
    HAS_LIGHTGBM = False

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    CatBoostRegressor = None
    HAS_CATBOOST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    shap = None
    HAS_SHAP = False

# -----------------------------
# Main project settings
# -----------------------------
RANDOM_STATE = 42
TARGET = "Rut_20k"
GROUP_COL = "MixDesignKey"

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.20

# Keep this False while developing and selecting the model.
# Change to True only for the final, one-time locked-test evaluation.
REVEAL_LOCKED_TEST = False

# Fast mode for checking that the notebook runs.
# For final research results, set QUICK_RUN = False and raise tuning iterations.
QUICK_RUN = True
CV_FOLDS = 5
N_TARGET_BINS = 5
N_ITER_FAST = 10
N_ITER_FINAL = 60
N_ITER_TUNE = N_ITER_FAST if QUICK_RUN else N_ITER_FINAL
N_JOBS = -1

# Select the main feature set used by most cells.
FEATURE_SET_NAME = "VolumetricsB_NoADT_RBR_Both"

# Output folders for tables, models, and figures.
OUTPUT_DIR = Path("/content/Rut20k_Reorganized_Colab_outputs") if Path("/content").exists() else Path("Rut20k_Reorganized_Colab_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
SPLIT_DIR = OUTPUT_DIR / "splits"
for folder in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, SPLIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)

print(f"Target: {TARGET}")
print(f"Group variable: {GROUP_COL}")
print(f"Split: {int(TRAIN_SIZE*100)}/{int(VALIDATION_SIZE*100)}/{int(TEST_SIZE*100)} train/validation/locked-test")
print(f"Quick run: {QUICK_RUN}; tuning iterations per model: {N_ITER_TUNE}")
print(f"XGBoost={HAS_XGBOOST}, LightGBM={HAS_LIGHTGBM}, CatBoost={HAS_CATBOOST}, SHAP={HAS_SHAP}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")

## 1.1 Continued — Load the Dataset

Upload or point to your cleaned rutting dataset, usually `Rutting_Cleaned_with_RBR.xlsx`.  
The helper functions below clean column names, map common column aliases, and create important RBR/interaction variables when possible.

In [ ]:
# Change this path if your dataset has a different name or location.
DATA_PATH = Path("/content/Rutting_Cleaned_with_RBR.xlsx")
SHEET_CANDIDATES = ["Cleaned_With_RBR", "Cleaned_Dataset", "Cleaned_Data_Kept", "Sheet1", 0]

# Common aliases found across different versions of the cleaned asphalt files.
ALIASES = {
    "AsphaltContent_Design": ["AC_design", "AC_Design", "AsphaltContentDesign", "Design_AC"],
    "Pass4_75mm": ["P4.75", "P4_75", "Pass_4.75mm", "Pass_4_75mm", "Passing_4.75mm"],
    "Pass0_075mm": ["P0.075", "P0_075", "Pass_0.075mm", "Pass_0_075mm", "Passing_0.075mm"],
    "NMAS (mm)": ["NMAS", "NMAS_mm", "NMAS(mm)"],
    "PG_HighTemp": ["PG High", "PG_High", "PGHigh", "PG_High_Temp"],
    "RAP_pct": ["RAP", "RAP%", "RAP_Percent", "RAP_Pct"],
    "ACinRAP": ["AC_in_RAP", "AC_RAP", "ACin_RAP"],
    "Dust_Binder": ["DustBinder", "Dust_to_Binder", "Dust/Binder"],
    "SandEq": ["Sand_Equivalent", "SandEQ", "SE"],
    "DesignLev": ["DesignLevel", "Design_Level", "TrafficLevel"],
    "MixType": ["Mix_Type", "Mixture_Type", "Type"],
    "RAP_Class": ["RAPClass", "RAP class", "RAP_Classification"],
    "RBR_JMF_fraction": ["RBR_decimal", "RBR_fraction", "RBR"],
    "RBR_JMF_percent": ["RBR_percent", "RBR_pct"],
}

NUMERIC_HINTS = [
    "ACinRAP", "PG_HighTemp", "SandEq", "Dust_Binder", "VFA", "RAP_pct",
    "Pass4_75mm", "Pass0_075mm", "FAA", "CAA", "Absorption", "VMA", "Va",
    "Gmb", "Gmm", "Gsb", "AsphaltContent_Design", "RAP_pct_x_ACinRAP",
    "NMAS (mm)", "RBR_JMF_fraction", "RBR_JMF_percent", "RAP_Binder_Contribution",
    "PG_x_RBR", "PG_x_RAPAC", "Abs_x_RBR", "SandEq_x_DustBinder",
    "Va_x_Gmm", "VFA_x_AC", "P0075_x_DustBinder",
]
CATEGORICAL_HINTS = ["MixType", "DesignLev", "RAP_Class"]

# Columns that should not be used as predictors because they are IDs, targets, timestamps,
# flags, record identifiers, or post-outcome/leakage variables.
DROP_ALWAYS = [
    GROUP_COL, "Rut_20k", "SCB", "LWT_Record_ID", "SCB_Record_ID", "LWT_LastUpdated",
    "SCB_LastUpdated", "Rut_Replicate_Number", "SCB_Replicate_Number", "Predictor_Source",
    "Gmm_record_date", "Gmb_record_date", "Gmb_specimen_AC", "Review_Flags",
    "Aggregate_components_used", "IsLeft", "ADT", "RBR_band",
    "Flag_RBR_out_of_range", "Flag_missing_input",
]


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Strip extra spaces from column names so later code can match them reliably."""
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def copy_alias_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Create standard column names from known alternative names without deleting originals."""
    out = df.copy()
    for canonical, aliases in ALIASES.items():
        if canonical in out.columns:
            continue
        for alias in aliases:
            if alias in out.columns:
                out[canonical] = out[alias]
                break
    return out


def parse_adt_to_ordinal(value: Any) -> float:
    """Convert ADT text ranges into a numeric midpoint or a simple low/medium/high ordinal."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip().lower().replace(",", "")
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", text)]
    if len(nums) >= 2:
        return float(np.mean(nums[:2]))
    if len(nums) == 1:
        return nums[0]
    if "low" in text:
        return 1.0
    if "medium" in text or "med" in text:
        return 2.0
    if "high" in text:
        return 3.0
    return np.nan


def safe_divide(a: Any, b: Any) -> pd.Series:
    """Numerically safe division that returns NaN when the denominator is zero or missing."""
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce").replace(0, np.nan)
    return a / b


def create_engineered_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Create physically meaningful asphalt mixture features used later by the models."""
    out = df.copy()

    if "ADT" in out.columns and "ADT_DOTD_ord" not in out.columns:
        out["ADT_DOTD_ord"] = out["ADT"].apply(parse_adt_to_ordinal)

    # RAP_pct_x_ACinRAP is a traditional proxy for RAP binder contribution.
    if {"RAP_pct", "ACinRAP"}.issubset(out.columns):
        out["RAP_pct_x_ACinRAP"] = pd.to_numeric(out["RAP_pct"], errors="coerce") * pd.to_numeric(out["ACinRAP"], errors="coerce")

    # True RBR = RAP binder contribution / design binder content.
    # Prefer the cleaned-file RBR columns if they exist; otherwise recompute defensively.
    if "RBR_JMF_fraction" not in out.columns and {"RAP_pct", "ACinRAP", "AsphaltContent_Design"}.issubset(out.columns):
        rap_binder_contribution = pd.to_numeric(out["RAP_pct"], errors="coerce") * pd.to_numeric(out["ACinRAP"], errors="coerce") / 100.0
        out["RAP_Binder_Contribution"] = rap_binder_contribution
        out["RBR_JMF_fraction"] = safe_divide(rap_binder_contribution, out["AsphaltContent_Design"])

    if "RBR_JMF_percent" not in out.columns and "RBR_JMF_fraction" in out.columns:
        out["RBR_JMF_percent"] = 100.0 * pd.to_numeric(out["RBR_JMF_fraction"], errors="coerce")

    for col in ["RBR_JMF_fraction", "RBR_JMF_percent"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    # Physics-sensible interaction features.
    def has(*cols: str) -> bool:
        return set(cols).issubset(out.columns)

    if has("PG_HighTemp", "RBR_JMF_fraction"):
        out["PG_x_RBR"] = pd.to_numeric(out["PG_HighTemp"], errors="coerce") * pd.to_numeric(out["RBR_JMF_fraction"], errors="coerce")
    if has("PG_HighTemp", "RAP_pct_x_ACinRAP"):
        out["PG_x_RAPAC"] = pd.to_numeric(out["PG_HighTemp"], errors="coerce") * pd.to_numeric(out["RAP_pct_x_ACinRAP"], errors="coerce")
    if has("Absorption", "RBR_JMF_fraction"):
        out["Abs_x_RBR"] = pd.to_numeric(out["Absorption"], errors="coerce") * pd.to_numeric(out["RBR_JMF_fraction"], errors="coerce")
    if has("SandEq", "Dust_Binder"):
        out["SandEq_x_DustBinder"] = pd.to_numeric(out["SandEq"], errors="coerce") * pd.to_numeric(out["Dust_Binder"], errors="coerce")
    if has("Va", "Gmm"):
        out["Va_x_Gmm"] = pd.to_numeric(out["Va"], errors="coerce") * pd.to_numeric(out["Gmm"], errors="coerce")
    if has("VFA", "AsphaltContent_Design"):
        out["VFA_x_AC"] = pd.to_numeric(out["VFA"], errors="coerce") * pd.to_numeric(out["AsphaltContent_Design"], errors="coerce")
    if has("Pass0_075mm", "Dust_Binder"):
        out["P0075_x_DustBinder"] = pd.to_numeric(out["Pass0_075mm"], errors="coerce") * pd.to_numeric(out["Dust_Binder"], errors="coerce")

    # RBR band is useful for diagnostics, but it is not used as a predictor.
    if "RBR_band" not in out.columns and "RBR_JMF_fraction" in out.columns:
        f = pd.to_numeric(out["RBR_JMF_fraction"], errors="coerce")
        out["RBR_band"] = pd.cut(
            f,
            bins=[-0.001, 0.0001, 0.15, 0.25, np.inf],
            labels=["No RAP (0)", "Low (>0-0.15)", "Moderate (>0.15-0.25)", "High (>0.25)"],
        ).astype("object")
    return out


def read_excel_best_sheet(path: Path) -> pd.DataFrame:
    """Read the most likely cleaned sheet; if it is not found, read the first sheet."""
    xls = pd.ExcelFile(path)
    for sheet in SHEET_CANDIDATES:
        if sheet == 0:
            continue
        if str(sheet) in xls.sheet_names:
            print(f"Using sheet: {sheet}")
            return pd.read_excel(path, sheet_name=sheet)
    print(f"Using first sheet: {xls.sheet_names[0]}")
    return pd.read_excel(path, sheet_name=xls.sheet_names[0])


def load_dataset(path: Path) -> pd.DataFrame:
    """Load, clean, map aliases, engineer features, and remove rows with missing target."""
    if not path.exists():
        try:
            from google.colab import files
            print("Dataset not found at DATA_PATH. Please upload the Excel file now.")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No file was uploaded.")
            path = Path(next(iter(uploaded.keys())))
        except Exception as exc:
            raise FileNotFoundError(
                f"Could not find {path}. Upload the file to Colab or change DATA_PATH."
            ) from exc

    df_raw = read_excel_best_sheet(path)
    df = clean_column_names(df_raw)
    df = copy_alias_columns(df)
    df = create_engineered_columns(df)

    if TARGET not in df.columns:
        raise KeyError(f"Target column {TARGET!r} was not found. Available columns include: {list(df.columns)[:30]}")
    if GROUP_COL not in df.columns:
        raise KeyError(f"Group column {GROUP_COL!r} was not found. It is required for leakage-safe splitting.")

    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = df.loc[df[TARGET].notna()].reset_index(drop=True)
    return df


df = load_dataset(DATA_PATH)

print("\nDataset audit preview")
print("Rows, columns:", df.shape)
print("\nColumn names:")
print(list(df.columns))
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nFirst rows:")
display(df.head())

## 1.2 Missing Values Check

This cell reports missing values by column, missing-value percentages, and a recommended action.  
The decision is not automatic deletion; the goal is to identify whether missingness is small, systematic, or severe.

In [ ]:
def missing_value_report(data: pd.DataFrame) -> pd.DataFrame:
    """Create a missing-value table with a suggested review action."""
    report = pd.DataFrame({
        "Missing_Count": data.isna().sum(),
        "Missing_Percent": 100.0 * data.isna().mean(),
        "Dtype": data.dtypes.astype(str),
    }).sort_values("Missing_Percent", ascending=False)

    def action(pct: float) -> str:
        if pct == 0:
            return "No action needed."
        if pct < 5:
            return "Low missingness: impute inside pipeline or verify source records."
        if pct < 30:
            return "Moderate missingness: investigate pattern; impute only if engineering meaning is clear."
        return "High missingness: consider dropping feature or using only after source verification."

    report["Recommended_Action"] = report["Missing_Percent"].apply(action)
    return report

missing_report = missing_value_report(df)
display(missing_report)
missing_report.to_csv(TABLE_DIR / "missing_values_report.csv")

print("Summary:")
print(f"Columns with at least one missing value: {(missing_report['Missing_Count'] > 0).sum()} of {df.shape[1]}")
print("Important note: model pipelines below impute missing values using training data only.")

## 1.3 Abnormal Engineering Range Check

This cell checks whether important asphalt mixture variables fall inside broad engineering review ranges.  
These limits are **screening thresholds**, not automatic deletion rules. Values outside the range should be checked against original lab/JMF records.

In [ ]:
# Broad engineering review ranges. Adjust these to match your DOTD specification table if needed.
ENGINEERING_RANGES = {
    "AsphaltContent_Design": (3.0, 8.5, "Design binder content outside a typical asphalt mixture range."),
    "RAP_pct": (0.0, 60.0, "Very high RAP may be possible but requires verification."),
    "ACinRAP": (0.0, 8.0, "AC in RAP outside expected binder-content range."),
    "RBR_JMF_fraction": (0.0, 1.0, "RBR fraction should normally be between 0 and 1."),
    "RBR_JMF_percent": (0.0, 100.0, "RBR percent should normally be between 0 and 100."),
    "PG_HighTemp": (52.0, 82.0, "Binder high PG appears outside common PG high-temperature grades."),
    "Va": (0.0, 10.0, "Air voids are unusually low/high and should be checked."),
    "VMA": (8.0, 30.0, "VMA appears outside a broad asphalt-mixture range."),
    "VFA": (40.0, 90.0, "VFA appears unusually low/high and should be checked."),
    "Gmm": (2.0, 3.0, "Gmm specific gravity outside expected range."),
    "Gmb": (2.0, 3.0, "Gmb specific gravity outside expected range."),
    "Gsb": (2.0, 3.2, "Aggregate bulk specific gravity outside expected range."),
    "NMAS (mm)": (4.75, 37.5, "NMAS outside common asphalt mixture sizes."),
    "Dust_Binder": (0.3, 2.0, "Dust-to-binder ratio may affect stiffness, durability, and rutting."),
    "SandEq": (20.0, 100.0, "Sand equivalent outside expected quality range."),
    "Absorption": (0.0, 5.0, "Aggregate absorption unusually high/low."),
    "Pass4_75mm": (0.0, 100.0, "Percent passing must be between 0 and 100."),
    "Pass0_075mm": (0.0, 15.0, "P200/fines content unusually high for many dense-graded mixes."),
    "FAA": (0.0, 100.0, "Fine aggregate angularity should be a percent-like value."),
    "CAA": (0.0, 100.0, "Coarse aggregate angularity should be a percent-like value."),
}


def engineering_range_check(data: pd.DataFrame, ranges: Dict[str, Tuple[float, float, str]]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Flag rows outside broad engineering review ranges."""
    summary_rows = []
    flagged_records = []

    for col, (lower, upper, reason) in ranges.items():
        if col not in data.columns:
            continue
        values = pd.to_numeric(data[col], errors="coerce")
        mask = values.notna() & ((values < lower) | (values > upper))
        summary_rows.append({
            "Variable": col,
            "Lower_Bound": lower,
            "Upper_Bound": upper,
            "Flagged_Count": int(mask.sum()),
            "Flagged_Percent": 100.0 * float(mask.mean()),
            "Reason": reason,
        })
        if mask.any():
            temp = data.loc[mask, [GROUP_COL, TARGET, col]].copy()
            temp["Variable"] = col
            temp["Value"] = temp[col]
            temp["Lower_Bound"] = lower
            temp["Upper_Bound"] = upper
            temp["Engineering_Reason"] = reason
            flagged_records.append(temp[[GROUP_COL, TARGET, "Variable", "Value", "Lower_Bound", "Upper_Bound", "Engineering_Reason"]])

    summary = pd.DataFrame(summary_rows).sort_values("Flagged_Count", ascending=False)
    flags = pd.concat(flagged_records, ignore_index=True) if flagged_records else pd.DataFrame()
    return summary, flags

engineering_summary, engineering_flags = engineering_range_check(df, ENGINEERING_RANGES)

display(engineering_summary)
if not engineering_flags.empty:
    print("Flagged examples:")
    display(engineering_flags.head(30))
else:
    print("No abnormal values found using the current broad engineering ranges.")

engineering_summary.to_csv(TABLE_DIR / "engineering_range_summary.csv", index=False)
engineering_flags.to_csv(TABLE_DIR / "engineering_range_flagged_rows.csv", index=False)

print("Engineering interpretation note:")
print("Flagged rows should be verified before deletion. Some may be valid extreme mixtures; others may be unit, entry, or merge errors.")

## 1.4 Duplicate and Replicate `MixDesignKey` Check

This cell separates exact duplicate rows from repeated mix designs.  
Repeated `MixDesignKey` values may be true lab replicates, repeated test measurements, or accidental duplication. They should stay in the same split.

In [ ]:
# Exact duplicate rows across all columns.
exact_duplicate_mask = df.duplicated(keep=False)
exact_duplicates = df.loc[exact_duplicate_mask].copy()

# Repeated MixDesignKey values indicate replicate tests or repeated records for the same mix design.
replicate_counts = (
    df.groupby(GROUP_COL)
      .size()
      .reset_index(name="Rows_Per_MixDesignKey")
      .sort_values("Rows_Per_MixDesignKey", ascending=False)
)
replicated_keys = replicate_counts.loc[replicate_counts["Rows_Per_MixDesignKey"] > 1]

print(f"Exact duplicate rows: {len(exact_duplicates)}")
print(f"MixDesignKey values with repeated rows: {len(replicated_keys)}")
print(f"Maximum rows for one MixDesignKey: {replicate_counts['Rows_Per_MixDesignKey'].max()}")

display(replicated_keys.head(20))

if len(exact_duplicates) > 0:
    print("Exact duplicate examples:")
    display(exact_duplicates.head())

replicate_counts.to_csv(TABLE_DIR / "mixdesignkey_replicate_counts.csv", index=False)
exact_duplicates.to_csv(TABLE_DIR / "exact_duplicate_rows.csv", index=False)

print("Interpretation:")
print("Replicates are not automatically wrong. The leakage-safe split below groups by MixDesignKey so replicate information cannot leak across train/validation/test.")

## 1.5 Target Distribution

This cell examines `Rut_20k` distribution using descriptive statistics, histogram, boxplot, and skewness.  
A strongly skewed target may justify a log transformation, but the final decision should consider engineering meaning and validation performance.

In [ ]:
y_all = pd.to_numeric(df[TARGET], errors="coerce")

target_stats = y_all.describe().to_frame("Rut_20k_Stats")
target_skewness = y_all.skew()

display(target_stats)
print(f"Target skewness: {target_skewness:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(y_all.dropna(), bins=30, edgecolor="black")
ax.set_title(f"Distribution of {TARGET}")
ax.set_xlabel(f"{TARGET} (rut depth, mm)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig(FIG_DIR / "target_histogram.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.boxplot(y_all.dropna(), vert=False)
ax.set_title(f"Boxplot of {TARGET}")
ax.set_xlabel(f"{TARGET} (rut depth, mm)")
plt.tight_layout()
plt.savefig(FIG_DIR / "target_boxplot.png", dpi=200)
plt.show()

if abs(target_skewness) > 1:
    print("Possible action: target is strongly skewed; compare raw-target and log1p-target models using validation/OOF scores.")
else:
    print("Possible action: target skewness is not severe; raw-scale modeling may be acceptable.")

## 1.6 Outlier Screening — IQR Method

This cell calculates Q1, Q3, IQR, lower bound, and upper bound for selected important variables.  
The IQR method is univariate; it can flag extreme values one variable at a time but does not capture multivariable abnormal combinations.

In [ ]:
AUDIT_VARIABLES = [
    TARGET, "PG_HighTemp", "RBR_JMF_fraction", "RAP_pct", "ACinRAP",
    "AsphaltContent_Design", "Va", "VMA", "VFA", "Gmm", "Gmb",
    "Dust_Binder", "Pass4_75mm", "Pass0_075mm", "SandEq", "Absorption",
]
AUDIT_VARIABLES = [c for c in AUDIT_VARIABLES if c in df.columns]


def iqr_outlier_report(data: pd.DataFrame, columns: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate IQR bounds and return both a variable-level summary and row-level flags."""
    summary_rows = []
    row_flags = []
    for col in columns:
        values = pd.to_numeric(data[col], errors="coerce")
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = values.notna() & ((values < lower) | (values > upper))
        summary_rows.append({
            "Variable": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower_Bound": lower,
            "Upper_Bound": upper,
            "Outlier_Count": int(mask.sum()),
            "Outlier_Percent": 100.0 * float(mask.mean()),
        })
        if mask.any():
            temp = data.loc[mask, [GROUP_COL, TARGET, col]].copy()
            temp["Variable"] = col
            temp["Value"] = temp[col]
            temp["IQR_Lower_Bound"] = lower
            temp["IQR_Upper_Bound"] = upper
            row_flags.append(temp[[GROUP_COL, TARGET, "Variable", "Value", "IQR_Lower_Bound", "IQR_Upper_Bound"]])
    summary = pd.DataFrame(summary_rows).sort_values("Outlier_Count", ascending=False)
    flags = pd.concat(row_flags, ignore_index=True) if row_flags else pd.DataFrame()
    return summary, flags

iqr_summary, iqr_flags = iqr_outlier_report(df, AUDIT_VARIABLES)
display(iqr_summary)
if not iqr_flags.empty:
    display(iqr_flags.head(30))

iqr_summary.to_csv(TABLE_DIR / "iqr_outlier_summary.csv", index=False)
iqr_flags.to_csv(TABLE_DIR / "iqr_outlier_rows.csv", index=False)

# Boxplots for important variables.
for col in AUDIT_VARIABLES:
    values = pd.to_numeric(df[col], errors="coerce").dropna()
    if len(values) < 2:
        continue
    fig, ax = plt.subplots(figsize=(7, 2.5))
    ax.boxplot(values, vert=False)
    ax.set_title(f"IQR boxplot — {col}")
    ax.set_xlabel(col)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"iqr_boxplot_{re.sub(r'[^A-Za-z0-9]+', '_', col)}.png", dpi=200)
    plt.show()

print("Important: IQR flags are screening results only. Do not remove them automatically without engineering verification.")

## 1.6 Continued — Isolation Forest Exploratory Screening

This cell uses Isolation Forest for exploratory multivariable outlier screening.  
Here it is fit on the full dataset only for data-audit visualization. In the modeling section, any outlier model or preprocessing decision must be fit only on training data.

In [ ]:
# Use selected numeric audit features, excluding the target, for exploratory anomaly screening.
iforest_features = [c for c in AUDIT_VARIABLES if c != TARGET]
X_iforest_raw = df[iforest_features].apply(pd.to_numeric, errors="coerce")

if len(iforest_features) >= 2:
    iforest_preprocess = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_iforest = iforest_preprocess.fit_transform(X_iforest_raw)

    iso = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=RANDOM_STATE,
    )
    iso.fit(X_iforest)

    # Higher anomaly score below means more abnormal after multiplying by -1.
    anomaly_score = -iso.score_samples(X_iforest)
    iso_label = iso.predict(X_iforest)  # -1 = anomaly, +1 = normal

    isolation_results = df[[GROUP_COL, TARGET]].copy()
    isolation_results["IsolationForest_AnomalyScore"] = anomaly_score
    isolation_results["IsolationForest_Flag"] = iso_label == -1

    # Compare Isolation Forest row flags with IQR row flags.
    iqr_flagged_index = set(iqr_flags.index) if not iqr_flags.empty else set()
    iqr_row_flag_mask = pd.Series(False, index=df.index)
    if not iqr_flags.empty:
        # A row is considered IQR-flagged if its MixDesignKey and target appear among IQR flags.
        flagged_pairs = set(zip(iqr_flags[GROUP_COL].astype(str), iqr_flags[TARGET].astype(float)))
        iqr_row_flag_mask = df.apply(lambda r: (str(r[GROUP_COL]), float(r[TARGET])) in flagged_pairs, axis=1)

    isolation_results["Any_IQR_Flag"] = iqr_row_flag_mask.values
    comparison = pd.crosstab(isolation_results["Any_IQR_Flag"], isolation_results["IsolationForest_Flag"], rownames=["Any_IQR_Flag"], colnames=["IsolationForest_Flag"])

    display(isolation_results.sort_values("IsolationForest_AnomalyScore", ascending=False).head(30))
    print("IQR vs Isolation Forest comparison:")
    display(comparison)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    order = np.argsort(anomaly_score)
    ax.plot(np.arange(len(order)), anomaly_score[order])
    ax.set_title("Isolation Forest anomaly scores")
    ax.set_xlabel("Rows sorted by anomaly score")
    ax.set_ylabel("Anomaly score; higher = more abnormal")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "isolation_forest_anomaly_scores.png", dpi=200)
    plt.show()

    isolation_results.to_csv(TABLE_DIR / "isolation_forest_exploratory_screening.csv", index=False)
else:
    print("Not enough numeric audit features for Isolation Forest screening.")

print("Interpretation: IQR detects univariate extremes; Isolation Forest detects unusual multivariable combinations. Neither method should automatically delete asphalt mixtures.")

# Step 2: Leakage-Safe Data Split

## 2.1 Define Features, Target, and Group Variable

This cell defines feature sets, removes leakage columns, identifies numeric/categorical features, and prepares `X`, `y`, and `groups`.

In [ ]:
# Feature sets based on the original workflow, with true RBR as a first-class feature.
RUT_BASE_NOADT = [
    "ACinRAP", "PG_HighTemp", "SandEq", "Dust_Binder", "VFA",
    "Pass4_75mm", "FAA", "Absorption", "VMA", "AsphaltContent_Design", "RAP_pct_x_ACinRAP",
]
VOLUMETRICS_B_NOADT = RUT_BASE_NOADT + ["NMAS (mm)", "Pass0_075mm", "Va", "Gmm", "CAA"]
VOLUMETRICS_B_RBR_BOTH = VOLUMETRICS_B_NOADT + ["RBR_JMF_fraction"]
VOLUMETRICS_B_RBR_REPLACE = [f for f in VOLUMETRICS_B_NOADT if f != "RAP_pct_x_ACinRAP"] + ["RBR_JMF_fraction"]
SHAP12_RBR = [
    "PG_HighTemp", "RBR_JMF_fraction", "SandEq", "Absorption", "VFA", "ACinRAP",
    "Dust_Binder", "Pass4_75mm", "Va", "Pass0_075mm", "Gmm", "FAA",
]
SHAP14_RBR = SHAP12_RBR + ["VMA", "AsphaltContent_Design"]
PHYSICAL_INTERACTIONS = [
    "PG_x_RBR", "PG_x_RAPAC", "Abs_x_RBR", "SandEq_x_DustBinder",
    "Va_x_Gmm", "VFA_x_AC", "P0075_x_DustBinder",
]
STRUCT_GRAD_DESIGN = RUT_BASE_NOADT + ["NMAS (mm)", "Pass0_075mm", "MixType", "DesignLev", "RAP_Class"]
FULL_EXPANDED_CLEAN = RUT_BASE_NOADT + [
    "NMAS (mm)", "Pass0_075mm", "MixType", "DesignLev", "RAP_Class",
    "CAA", "Gsb", "Va", "Gmm", "Gmb", "RBR_JMF_fraction",
]

FEATURE_SETS = {
    "VolumetricsB_NoADT_CurrentBest": VOLUMETRICS_B_NOADT,
    "VolumetricsB_NoADT_RBR_Both": VOLUMETRICS_B_RBR_BOTH,
    "VolumetricsB_NoADT_RBR_Replace_RAPAC": VOLUMETRICS_B_RBR_REPLACE,
    "SHAP12_RBR": SHAP12_RBR,
    "SHAP14_RBR": SHAP14_RBR,
    "SHAP12_RBR_PlusInteractions": SHAP12_RBR + PHYSICAL_INTERACTIONS,
    "SHAP14_RBR_PlusInteractions": SHAP14_RBR + PHYSICAL_INTERACTIONS,
    "StructGradDesign_CleanCategorical": STRUCT_GRAD_DESIGN,
    "FullExpanded_CleanCategorical": FULL_EXPANDED_CLEAN,
}


def clean_categorical_series(s: pd.Series) -> pd.Series:
    """Convert missing or blank category values to a stable 'Missing' label."""
    return (
        s.astype("object")
        .where(s.notna(), "Missing")
        .astype(str)
        .str.strip()
        .replace({"": "Missing", "nan": "Missing", "None": "Missing"})
    )


def get_feature_matrix(data: pd.DataFrame, requested_features: List[str]) -> Tuple[pd.DataFrame, List[str], List[str], List[str], List[str]]:
    """Build X using available requested features and classify features as numeric or categorical."""
    unique_requested = []
    for f in requested_features:
        if f not in unique_requested:
            unique_requested.append(f)

    available = [f for f in unique_requested if f in data.columns and f not in DROP_ALWAYS]
    missing = [f for f in unique_requested if f not in data.columns]
    X = data[available].copy()

    numeric_features = []
    categorical_features = []
    for col in X.columns:
        if col in CATEGORICAL_HINTS:
            categorical_features.append(col)
            X[col] = clean_categorical_series(X[col])
        elif col in NUMERIC_HINTS:
            numeric_features.append(col)
            X[col] = pd.to_numeric(X[col], errors="coerce").astype("float64")
        else:
            # Infer whether a feature is numeric by checking how often conversion succeeds.
            coerced = pd.to_numeric(X[col], errors="coerce")
            if coerced.notna().mean() >= 0.85:
                numeric_features.append(col)
                X[col] = coerced.astype("float64")
            else:
                categorical_features.append(col)
                X[col] = clean_categorical_series(X[col])

    return X, numeric_features, categorical_features, available, missing

selected_features = FEATURE_SETS[FEATURE_SET_NAME]
X, numeric_features, categorical_features, available_features, missing_features = get_feature_matrix(df, selected_features)
y = df[TARGET].astype(float).reset_index(drop=True)
groups = df[GROUP_COL].astype(str).reset_index(drop=True)

print(f"Feature set: {FEATURE_SET_NAME}")
print(f"Available features used ({len(available_features)}): {available_features}")
print(f"Missing requested features ({len(missing_features)}): {missing_features}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"X shape: {X.shape}; y length: {len(y)}; unique groups: {groups.nunique()}")

## 2.2 Group-Based 70/10/20 Split

This cell performs the leakage-safe split:

- 70% training
- 10% validation
- 20% locked test

The split is grouped by `MixDesignKey`, so the same mix design cannot appear in more than one split.

In [ ]:
def make_target_bins(y_values: pd.Series, n_bins: int = N_TARGET_BINS) -> pd.Series:
    """Create quantile target bins for approximate stratification of a continuous target."""
    y_values = pd.Series(y_values).reset_index(drop=True)
    for q in [n_bins, n_bins - 1, 4, 3, 2]:
        if q < 2:
            break
        try:
            bins = pd.qcut(y_values, q=q, labels=False, duplicates="drop")
            if bins.nunique(dropna=True) >= 2:
                return bins.astype(int)
        except Exception:
            continue
    return (y_values >= y_values.median()).astype(int)


def first_group_holdout(y_values: pd.Series, group_values: pd.Series, holdout_size: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    """Return keep and holdout indices using StratifiedGroupKFold when possible.

    If stratified grouping fails because of small group/bin counts, the function falls back to
    GroupShuffleSplit. The fallback remains group-safe, even if target stratification is weaker.
    """
    bins = make_target_bins(y_values)
    n = len(y_values)
    n_splits = max(2, int(round(1.0 / holdout_size)))

    try:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        keep_idx, holdout_idx = next(splitter.split(np.zeros(n), bins, group_values))
    except Exception as exc:
        print(f"StratifiedGroupKFold fallback used: {type(exc).__name__}: {exc}")
        splitter = GroupShuffleSplit(n_splits=1, test_size=holdout_size, random_state=seed)
        keep_idx, holdout_idx = next(splitter.split(np.zeros(n), y_values, groups=group_values))
    return np.array(keep_idx), np.array(holdout_idx)


def leakage_safe_70_10_20_split(X_data: pd.DataFrame, y_values: pd.Series, group_values: pd.Series) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Split into train/validation/locked-test while keeping groups separated."""
    # Stage 1: peel off the locked test first.
    dev_idx, test_idx = first_group_holdout(y_values, group_values, TEST_SIZE, RANDOM_STATE)

    # Stage 2: split the remaining development set into train and validation.
    val_fraction_of_dev = VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE)
    dev_y = y_values.iloc[dev_idx].reset_index(drop=True)
    dev_groups = group_values.iloc[dev_idx].reset_index(drop=True)
    train_pos, val_pos = first_group_holdout(dev_y, dev_groups, val_fraction_of_dev, RANDOM_STATE + 1)

    train_idx = dev_idx[train_pos]
    val_idx = dev_idx[val_pos]
    return train_idx, val_idx, test_idx


def check_group_leakage(group_values: pd.Series, train_idx: np.ndarray, val_idx: np.ndarray, test_idx: np.ndarray) -> Dict[str, int]:
    """Count overlapping groups between splits. All counts should be zero."""
    g_train = set(group_values.iloc[train_idx])
    g_val = set(group_values.iloc[val_idx])
    g_test = set(group_values.iloc[test_idx])
    return {
        "Train_Validation_Group_Overlap": len(g_train & g_val),
        "Train_Test_Group_Overlap": len(g_train & g_test),
        "Validation_Test_Group_Overlap": len(g_val & g_test),
    }

train_idx, val_idx, test_idx = leakage_safe_70_10_20_split(X, y, groups)
leakage_check = check_group_leakage(groups, train_idx, val_idx, test_idx)

X_train = X.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)

X_val = X.iloc[val_idx].reset_index(drop=True)
y_val = y.iloc[val_idx].reset_index(drop=True)
groups_val = groups.iloc[val_idx].reset_index(drop=True)

# Locked test is created but will not be used until Step 6.6.
X_locked_test = X.iloc[test_idx].reset_index(drop=True)
y_locked_test = y.iloc[test_idx].reset_index(drop=True)
groups_locked_test = groups.iloc[test_idx].reset_index(drop=True)

split_summary = pd.DataFrame([
    {"Split": "Train", "Rows": len(train_idx), "Percent": 100 * len(train_idx) / len(X), "Unique_MixDesignKey": groups_train.nunique(), "Target_Mean": y_train.mean(), "Target_SD": y_train.std()},
    {"Split": "Validation", "Rows": len(val_idx), "Percent": 100 * len(val_idx) / len(X), "Unique_MixDesignKey": groups_val.nunique(), "Target_Mean": y_val.mean(), "Target_SD": y_val.std()},
    {"Split": "Locked_Test", "Rows": len(test_idx), "Percent": 100 * len(test_idx) / len(X), "Unique_MixDesignKey": groups_locked_test.nunique(), "Target_Mean": y_locked_test.mean(), "Target_SD": y_locked_test.std()},
])

display(split_summary)
print("Group leakage check:", leakage_check)

if any(v != 0 for v in leakage_check.values()):
    raise RuntimeError("Group leakage detected. Stop and fix the split before modeling.")

split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)
df.iloc[train_idx].to_excel(SPLIT_DIR / "train_70pct_rows.xlsx", index=False)
df.iloc[val_idx].to_excel(SPLIT_DIR / "validation_10pct_rows.xlsx", index=False)
df.iloc[test_idx].to_excel(SPLIT_DIR / "LOCKED_test_20pct_DO_NOT_USE_FOR_TUNING.xlsx", index=False)

## 2.3 Locked Test Protection

The locked test set exists only as a protected holdout. Do not use it for tuning, feature selection, preprocessing decisions, SHAP/PDP model choice, or model selection.

In [ ]:
print("LOCKED TEST PROTECTION CHECK")
print("- The locked test was split first.")
print("- The locked test file was saved for traceability.")
print("- The locked test will not be scored unless REVEAL_LOCKED_TEST = True in Step 6.6.")
print("- Current setting:", REVEAL_LOCKED_TEST)
print("Locked test file:", SPLIT_DIR / "LOCKED_test_20pct_DO_NOT_USE_FOR_TUNING.xlsx")

## 2.4 Preprocessing Pipeline

This cell creates reusable preprocessing helpers.  
Numeric features are median-imputed. Categorical features are imputed and one-hot encoded. Scaling is added only for models that need it, such as linear regression, Lasso, Ridge, ElasticNet, distance-based models, or neural networks.

In [ ]:
def make_ohe() -> OneHotEncoder:
    """Create OneHotEncoder compatible with recent and older scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(num_cols: List[str], cat_cols: List[str]) -> ColumnTransformer:
    """Build preprocessing that is fit only inside the model pipeline."""
    transformers = []
    if num_cols:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        transformers.append(("num", numeric_pipe, num_cols))
    if cat_cols:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("onehot", make_ohe()),
        ])
        transformers.append(("cat", categorical_pipe, cat_cols))
    return ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=False)


def build_pipeline(model: Any, scale: bool = False) -> Pipeline:
    """Create a full leakage-safe pipeline: preprocessing + optional scaling + model."""
    steps = [("preprocess", build_preprocessor(numeric_features, categorical_features))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)


def make_train_cv_splits(y_values: pd.Series, group_values: pd.Series, n_splits: int = CV_FOLDS, seed: int = RANDOM_STATE):
    """Create group-safe CV splits for training-only model tuning and OOF prediction."""
    bins = make_target_bins(y_values)
    try:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(cv.split(np.zeros(len(y_values)), bins, groups=group_values))
        cv_name = f"StratifiedGroupKFold({n_splits}) by {GROUP_COL}"
    except Exception as exc:
        print(f"StratifiedGroupKFold failed; using GroupKFold. Reason: {type(exc).__name__}: {exc}")
        cv = GroupKFold(n_splits=n_splits)
        splits = list(cv.split(np.zeros(len(y_values)), y_values, groups=group_values))
        cv_name = f"GroupKFold({n_splits}) by {GROUP_COL}"
    return splits, cv_name

train_cv_splits, train_cv_name = make_train_cv_splits(y_train, groups_train)
print("Training-only CV:", train_cv_name)
print("Number of folds:", len(train_cv_splits))

# Common Modeling Helper Functions

The following helper functions keep model cells short and readable. They calculate R², RMSE, MAE, train-validation gap, and store every model result in one organized table.

In [ ]:
def rmse(y_true, y_pred) -> float:
    """Root mean squared error in the same unit as the target."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred, prefix: str = "") -> Dict[str, float]:
    """Return R2, RMSE, and MAE with an optional prefix."""
    return {
        f"{prefix}R2": float(r2_score(y_true, y_pred)),
        f"{prefix}RMSE": rmse(y_true, y_pred),
        f"{prefix}MAE": float(mean_absolute_error(y_true, y_pred)),
    }


def evaluate_fitted_model(model: Pipeline, model_name: str, X_tr: pd.DataFrame, y_tr: pd.Series, X_va: pd.DataFrame, y_va: pd.Series) -> Dict[str, Any]:
    """Evaluate a fitted model on training and validation data."""
    pred_train = model.predict(X_tr)
    pred_val = model.predict(X_va)
    train_m = regression_metrics(y_tr, pred_train, prefix="Train_")
    val_m = regression_metrics(y_va, pred_val, prefix="Validation_")
    row = {"Model": model_name, **train_m, **val_m}
    row["Train_Validation_R2_Gap"] = row["Train_R2"] - row["Validation_R2"]
    row["Overfit_Risk"] = "High" if row["Train_Validation_R2_Gap"] > 0.20 else "Moderate" if row["Train_Validation_R2_Gap"] > 0.10 else "Low"
    return row


def fit_simple_model(model_name: str, estimator: Any, scale: bool = False) -> Tuple[Pipeline, Dict[str, Any]]:
    """Fit a non-tuned model on the training set and evaluate on train/validation."""
    pipe = build_pipeline(estimator, scale=scale)
    pipe.fit(X_train, y_train)
    row = evaluate_fitted_model(pipe, model_name, X_train, y_train, X_val, y_val)
    return pipe, row


def tune_model(model_name: str, estimator: Any, param_distributions: Dict[str, List[Any]], scale: bool = False, n_iter: int = N_ITER_TUNE) -> Tuple[Pipeline, Dict[str, Any], pd.DataFrame]:
    """Tune a model using RandomizedSearchCV on training data only."""
    pipe = build_pipeline(estimator, scale=scale)
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=min(n_iter, max(1, math.prod([len(v) for v in param_distributions.values()]))),
        scoring="r2",
        cv=train_cv_splits,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        return_train_score=True,
        error_score=np.nan,
        verbose=0,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    row = evaluate_fitted_model(best_model, f"{model_name}_Tuned", X_train, y_train, X_val, y_val)
    row["Best_CV_R2_During_Tuning"] = float(search.best_score_)
    row["Best_Params"] = json.dumps(search.best_params_, default=str)
    cv_results = pd.DataFrame(search.cv_results_)
    return best_model, row, cv_results

# Containers used across the notebook.
RESULT_ROWS: List[Dict[str, Any]] = []
FITTED_MODELS: Dict[str, Pipeline] = {}
CV_RESULT_TABLES: Dict[str, pd.DataFrame] = {}


def register_model(name: str, fitted_model: Pipeline, row: Dict[str, Any], cv_results: Optional[pd.DataFrame] = None) -> None:
    """Store fitted model and result row in global containers."""
    FITTED_MODELS[name] = fitted_model
    row = row.copy()
    row["Stored_Model_Key"] = name
    RESULT_ROWS.append(row)
    if cv_results is not None:
        CV_RESULT_TABLES[name] = cv_results
    display(pd.DataFrame([row]).drop(columns=["Best_Params"], errors="ignore"))

# Step 3: Baseline Models

## 3.1 Linear Regression

Linear Regression is the simplest reference model. It is useful to check whether the target can be explained by a mostly linear relationship.

In [ ]:
linear_model, linear_row = fit_simple_model(
    "LinearRegression",
    LinearRegression(),
    scale=True,
)
register_model("LinearRegression", linear_model, linear_row)

print("Interpretation: If validation R2 is low, the relationship is likely nonlinear or missing key physical variables.")

## 3.2 Ridge Regression

Ridge adds L2 regularization, which helps stabilize linear models when predictors are correlated.

In [ ]:
ridge_model, ridge_row = fit_simple_model(
    "Ridge",
    Ridge(alpha=1.0, random_state=RANDOM_STATE),
    scale=True,
)
register_model("Ridge", ridge_model, ridge_row)

print("Interpretation: Ridge is useful as a regularized linear benchmark, especially when volumetric variables are correlated.")

## 3.3 Lasso Regression

Lasso adds L1 regularization and can shrink some coefficients to zero. It is a basic feature-selection baseline, but it may underfit nonlinear asphalt behavior.

In [ ]:
lasso_model, lasso_row = fit_simple_model(
    "Lasso",
    Lasso(alpha=0.001, max_iter=20000, random_state=RANDOM_STATE),
    scale=True,
)
register_model("Lasso", lasso_model, lasso_row)

print("Interpretation: If Lasso performs poorly, it suggests that simple sparse linear effects are not enough.")

## 3.4 ElasticNet Regression

ElasticNet combines L1 and L2 regularization. It is a stronger baseline when features are correlated and only some predictors are useful.

In [ ]:
elastic_model, elastic_row = fit_simple_model(
    "ElasticNet",
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
    scale=True,
)
register_model("ElasticNet", elastic_model, elastic_row)

print("Interpretation: ElasticNet is a fairer regularized linear baseline than plain Linear Regression.")

# Step 4: Main Machine Learning Models

Each model is first fit with reasonable defaults, then tuned using `RandomizedSearchCV` on the training set only.  
The validation set is used after tuning to compare models. The locked test is still untouched.

## 4.1 Random Forest

Random Forest is a bagging ensemble of decision trees. It handles nonlinear relationships and interactions but can overfit if trees are too deep and leaves are too small.

In [ ]:
rf_default, rf_default_row = fit_simple_model(
    "RandomForest_Default",
    RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=N_JOBS, min_samples_leaf=2),
    scale=False,
)
register_model("RandomForest_Default", rf_default, rf_default_row)

rf_params = {
    "model__n_estimators": [300, 500, 800],
    "model__max_depth": [None, 8, 12, 16],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
}
rf_tuned, rf_tuned_row, rf_cv = tune_model(
    "RandomForest",
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    rf_params,
    scale=False,
)
register_model("RandomForest_Tuned", rf_tuned, rf_tuned_row, rf_cv)

print("Inspect: max_depth, min_samples_leaf, max_features, and train-validation gap.")

## 4.2 Extra Trees

Extra Trees is similar to Random Forest but adds more randomization. It often reduces variance and can be robust for noisy tabular data.

In [ ]:
et_default, et_default_row = fit_simple_model(
    "ExtraTrees_Default",
    ExtraTreesRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=N_JOBS, min_samples_leaf=2),
    scale=False,
)
register_model("ExtraTrees_Default", et_default, et_default_row)

et_params = {
    "model__n_estimators": [300, 500, 800],
    "model__max_depth": [None, 8, 12, 16],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
}
et_tuned, et_tuned_row, et_cv = tune_model(
    "ExtraTrees",
    ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
    et_params,
    scale=False,
)
register_model("ExtraTrees_Tuned", et_tuned, et_tuned_row, et_cv)

print("Inspect: whether ExtraTrees lowers overfitting compared with Random Forest.")

## 4.3 XGBoost

XGBoost is a gradient boosting model. Important controls are learning rate, number of estimators, tree depth, minimum child weight, subsampling, and L1/L2 regularization.

In [ ]:
if HAS_XGBOOST:
    xgb_default, xgb_default_row = fit_simple_model(
        "XGBoost_Default",
        XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            n_estimators=500,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=20.0,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            tree_method="hist",
        ),
        scale=False,
    )
    register_model("XGBoost_Default", xgb_default, xgb_default_row)

    xgb_params = {
        "model__n_estimators": [400, 700, 1000],
        "model__learning_rate": [0.01, 0.02, 0.03, 0.05],
        "model__max_depth": [2, 3, 4],
        "model__min_child_weight": [5, 10, 20, 30],
        "model__subsample": [0.6, 0.75, 0.9],
        "model__colsample_bytree": [0.5, 0.7, 0.9],
        "model__gamma": [0.0, 0.1, 0.3],
        "model__reg_alpha": [0.0, 1.0, 3.0, 8.0],
        "model__reg_lambda": [10.0, 30.0, 60.0, 100.0],
    }
    xgb_tuned, xgb_tuned_row, xgb_cv = tune_model(
        "XGBoost",
        XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            tree_method="hist",
        ),
        xgb_params,
        scale=False,
    )
    register_model("XGBoost_Tuned", xgb_tuned, xgb_tuned_row, xgb_cv)
    print("Inspect: learning_rate, n_estimators, max_depth, reg_alpha, reg_lambda, and train-validation gap.")
else:
    print("XGBoost is not installed. Run the optional install cell first if you want this model.")

## 4.4 LightGBM

LightGBM is a fast gradient boosting model. Important controls are learning rate, number of estimators, number of leaves, minimum child samples, subsampling, and regularization.

In [ ]:
if HAS_LIGHTGBM:
    lgbm_default, lgbm_default_row = fit_simple_model(
        "LightGBM_Default",
        LGBMRegressor(
            n_estimators=600,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=20.0,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbose=-1,
        ),
        scale=False,
    )
    register_model("LightGBM_Default", lgbm_default, lgbm_default_row)

    lgbm_params = {
        "model__n_estimators": [400, 700, 1000],
        "model__learning_rate": [0.01, 0.02, 0.03, 0.05],
        "model__num_leaves": [7, 15, 31],
        "model__min_child_samples": [20, 40, 60, 80],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__reg_alpha": [0.0, 0.5, 1.0, 3.0],
        "model__reg_lambda": [5.0, 20.0, 50.0, 100.0],
    }
    lgbm_tuned, lgbm_tuned_row, lgbm_cv = tune_model(
        "LightGBM",
        LGBMRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1),
        lgbm_params,
        scale=False,
    )
    register_model("LightGBM_Tuned", lgbm_tuned, lgbm_tuned_row, lgbm_cv)
    print("Inspect: learning_rate, n_estimators, num_leaves, min_child_samples, and regularization.")
else:
    print("LightGBM is not installed. Run the optional install cell first if you want this model.")

## 4.5 CatBoost

CatBoost is strong for tabular data and categorical variables. In this pipeline, categorical features are one-hot encoded for consistency with other models.  
If you want native categorical handling, use CatBoost directly with `cat_features`, but keep the same train/validation/test split.

In [ ]:
if HAS_CATBOOST:
    cat_default, cat_default_row = fit_simple_model(
        "CatBoost_Default",
        CatBoostRegressor(
            iterations=600,
            learning_rate=0.03,
            depth=4,
            l2_leaf_reg=20.0,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=0,
        ),
        scale=False,
    )
    register_model("CatBoost_Default", cat_default, cat_default_row)

    cat_params = {
        "model__iterations": [400, 700, 1000],
        "model__learning_rate": [0.01, 0.02, 0.03, 0.05],
        "model__depth": [3, 4, 5, 6],
        "model__l2_leaf_reg": [5.0, 10.0, 20.0, 50.0, 100.0],
        "model__random_strength": [0.5, 1.0, 2.0, 4.0],
    }
    cat_tuned, cat_tuned_row, cat_cv = tune_model(
        "CatBoost",
        CatBoostRegressor(loss_function="RMSE", random_seed=RANDOM_STATE, verbose=0),
        cat_params,
        scale=False,
    )
    register_model("CatBoost_Tuned", cat_tuned, cat_tuned_row, cat_cv)
    print("Inspect: iterations, learning_rate, depth, l2_leaf_reg, and train-validation gap.")
else:
    print("CatBoost is not installed. Run the optional install cell first if you want this model.")

## 4.6 HistGradientBoosting

HistGradientBoosting is scikit-learn's efficient gradient boosting model. It is useful as a strong dependency-light comparison model.

In [ ]:
hgb_default, hgb_default_row = fit_simple_model(
    "HistGradientBoosting_Default",
    HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.03,
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
    scale=False,
)
register_model("HistGradientBoosting_Default", hgb_default, hgb_default_row)

hgb_params = {
    "model__max_iter": [300, 500, 800],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08],
    "model__max_leaf_nodes": [7, 15, 31],
    "model__min_samples_leaf": [20, 40, 60],
    "model__l2_regularization": [0.0, 0.1, 1.0, 5.0],
}
hgb_tuned, hgb_tuned_row, hgb_cv = tune_model(
    "HistGradientBoosting",
    HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    hgb_params,
    scale=False,
)
register_model("HistGradientBoosting_Tuned", hgb_tuned, hgb_tuned_row, hgb_cv)

print("Inspect: learning_rate, max_iter, max_leaf_nodes, min_samples_leaf, and l2_regularization.")

## 4.7 Main Model Comparison Table

This table compares all baseline and main models trained so far.  
The key warning sign is a large train-validation gap: high training R² with much lower validation R² suggests overfitting.

In [ ]:
results_df = pd.DataFrame(RESULT_ROWS).sort_values("Validation_R2", ascending=False).reset_index(drop=True)

display(results_df.drop(columns=["Best_Params"], errors="ignore"))
results_df.to_csv(TABLE_DIR / "model_results_train_validation.csv", index=False)

print("Best model by validation R2 so far:")
best_so_far = results_df.iloc[0]
print(best_so_far[["Stored_Model_Key", "Model", "Train_R2", "Validation_R2", "Train_Validation_R2_Gap", "Overfit_Risk"]])

# Step 5: Advanced Model Comparison

These cells are optional. They must follow the same rules: train/development only for comparison, validation for selection, and locked test only once at the final stage.

## 5.1 Optional TabPFN

TabPFN can be useful for small-to-medium tabular datasets, but installation, GPU/CPU runtime, and package version compatibility may limit feasibility. This cell is disabled by default.

In [ ]:
RUN_TABPFN = False

if RUN_TABPFN:
    try:
        from tabpfn import TabPFNRegressor

        tabpfn_preprocess = build_preprocessor(numeric_features, categorical_features)
        X_train_tab = tabpfn_preprocess.fit_transform(X_train)
        X_val_tab = tabpfn_preprocess.transform(X_val)

        tabpfn_model = TabPFNRegressor()
        tabpfn_model.fit(X_train_tab, y_train.to_numpy())
        pred_train = tabpfn_model.predict(X_train_tab)
        pred_val = tabpfn_model.predict(X_val_tab)

        row = {
            "Model": "TabPFN_Optional",
            **regression_metrics(y_train, pred_train, prefix="Train_"),
            **regression_metrics(y_val, pred_val, prefix="Validation_"),
            "Stored_Model_Key": "TabPFN_Optional",
        }
        row["Train_Validation_R2_Gap"] = row["Train_R2"] - row["Validation_R2"]
        RESULT_ROWS.append(row)
        display(pd.DataFrame([row]))
    except Exception as exc:
        print(f"TabPFN was not feasible in this environment: {type(exc).__name__}: {exc}")
else:
    print("TabPFN optional cell skipped. Set RUN_TABPFN = True after installing tabpfn.")

## 5.2 Optional AutoGluon Tabular

AutoGluon is useful for automated tabular model comparison. It is heavier than scikit-learn and may take longer to install/run in Colab. This cell uses only train and validation data.

In [ ]:
RUN_AUTOGLUON = False

if RUN_AUTOGLUON:
    try:
        from autogluon.tabular import TabularPredictor

        train_ag = X_train.copy()
        train_ag[TARGET] = y_train.values
        val_ag = X_val.copy()
        val_ag[TARGET] = y_val.values

        ag_path = str(OUTPUT_DIR / "autogluon_models")
        predictor = TabularPredictor(label=TARGET, problem_type="regression", eval_metric="r2", path=ag_path)
        predictor.fit(train_data=train_ag, tuning_data=val_ag, presets="medium_quality", time_limit=900)

        pred_train = predictor.predict(train_ag.drop(columns=[TARGET]))
        pred_val = predictor.predict(val_ag.drop(columns=[TARGET]))
        row = {
            "Model": "AutoGluon_Optional",
            **regression_metrics(y_train, pred_train, prefix="Train_"),
            **regression_metrics(y_val, pred_val, prefix="Validation_"),
            "Stored_Model_Key": "AutoGluon_Optional",
        }
        row["Train_Validation_R2_Gap"] = row["Train_R2"] - row["Validation_R2"]
        RESULT_ROWS.append(row)
        display(pd.DataFrame([row]))
        display(predictor.leaderboard(val_ag, silent=True))
    except Exception as exc:
        print(f"AutoGluon was not feasible in this environment: {type(exc).__name__}: {exc}")
else:
    print("AutoGluon optional cell skipped. Set RUN_AUTOGLUON = True after installing autogluon.tabular.")

## 5.3 Optional TabM / RealMLP Family

TabM and RealMLP-style models may require specialized packages and sometimes GPU acceleration.  
Keep them optional and compare them fairly using the same train/validation split and no locked-test access.

In [ ]:
RUN_TABM_REALMLP = False

if RUN_TABM_REALMLP:
    print("Optional TabM/RealMLP block enabled.")
    print("Install the package required by your chosen implementation, then add the model here.")
    print("Use the same X_train, y_train, X_val, y_val and never use X_locked_test for selection.")
else:
    print("TabM/RealMLP optional cell skipped. Keep it optional unless the package and runtime are available.")

# Step 6: Validation Framework

This section goes beyond a single train/validation split.  
It computes training score, out-of-fold score, validation score, repeated CV, nested CV, and final locked-test evaluation.

## 6.1 Training Score and 6.3 Validation Score

The table below already contains training and validation scores for all registered models. Training score alone is not reliable because it can be inflated by overfitting.

In [ ]:
results_df = pd.DataFrame(RESULT_ROWS).sort_values("Validation_R2", ascending=False).reset_index(drop=True)
display(results_df.drop(columns=["Best_Params"], errors="ignore"))

print("Training score answers: how well the model fits data it already saw.")
print("Validation score answers: how well the model performs on held-out development data used for model selection.")

## 6.2 Out-of-Fold Score

OOF predictions are generated by fitting the model on training folds and predicting the unseen fold.  
OOF is more reliable than training score because every prediction is made for a row not used to fit that fold model.

In [ ]:
def oof_predictions(model: Pipeline, X_data: pd.DataFrame, y_values: pd.Series, group_values: pd.Series, n_splits: int = CV_FOLDS) -> Tuple[np.ndarray, pd.DataFrame]:
    """Generate group-safe out-of-fold predictions for one model."""
    splits, cv_name = make_train_cv_splits(y_values.reset_index(drop=True), group_values.reset_index(drop=True), n_splits=n_splits)
    oof = np.full(len(y_values), np.nan, dtype=float)
    fold_rows = []
    for fold, (tr, va) in enumerate(splits, start=1):
        est = clone(model)
        est.fit(X_data.iloc[tr], y_values.iloc[tr])
        pred = est.predict(X_data.iloc[va])
        oof[va] = pred
        fold_metric = regression_metrics(y_values.iloc[va], pred, prefix="")
        fold_rows.append({"Fold": fold, "CV": cv_name, **fold_metric})
    return oof, pd.DataFrame(fold_rows)

# Calculate OOF for the current best validation model.
best_key_by_val = results_df.iloc[0]["Stored_Model_Key"]
best_model_by_val = FITTED_MODELS.get(best_key_by_val)

if best_model_by_val is not None:
    oof_pred, oof_fold_table = oof_predictions(best_model_by_val, X_train, y_train, groups_train)
    oof_metric = regression_metrics(y_train, oof_pred, prefix="OOF_")
    display(pd.DataFrame([{"Model_Key": best_key_by_val, **oof_metric}]))
    display(oof_fold_table)
    oof_fold_table.to_csv(TABLE_DIR / f"oof_folds_{best_key_by_val}.csv", index=False)
else:
    print("Best validation model is not stored in FITTED_MODELS, likely because it came from an optional external framework.")

## 6.4 Repeated Cross-Validation

Repeated CV estimates stability by repeating group-safe CV with different random seeds.  
The reported mean ± standard deviation is more informative than one split alone.

In [ ]:
def repeated_group_cv(model: Pipeline, X_data: pd.DataFrame, y_values: pd.Series, group_values: pd.Series, repeats: int = 3) -> pd.DataFrame:
    """Repeated group-safe CV. Each repeat changes the random seed for StratifiedGroupKFold."""
    rows = []
    for rep in range(1, repeats + 1):
        splits, cv_name = make_train_cv_splits(
            y_values.reset_index(drop=True),
            group_values.reset_index(drop=True),
            n_splits=CV_FOLDS,
            seed=RANDOM_STATE + 100 * rep,
        )
        for fold, (tr, va) in enumerate(splits, start=1):
            est = clone(model)
            est.fit(X_data.iloc[tr], y_values.iloc[tr])
            pred = est.predict(X_data.iloc[va])
            m = regression_metrics(y_values.iloc[va], pred, prefix="")
            rows.append({"Repeat": rep, "Fold": fold, "CV": cv_name, **m})
    return pd.DataFrame(rows)

REPEATED_CV_REPEATS = 2 if QUICK_RUN else 5

if best_model_by_val is not None:
    repeated_cv_table = repeated_group_cv(best_model_by_val, X_train, y_train, groups_train, repeats=REPEATED_CV_REPEATS)
    repeated_summary = repeated_cv_table[["R2", "RMSE", "MAE"]].agg(["mean", "std"]).T.reset_index()
    repeated_summary.columns = ["Metric", "Mean", "SD"]
    repeated_summary.insert(0, "Model_Key", best_key_by_val)
    display(repeated_summary)
    repeated_cv_table.to_csv(TABLE_DIR / f"repeated_cv_folds_{best_key_by_val}.csv", index=False)
    repeated_summary.to_csv(TABLE_DIR / f"repeated_cv_summary_{best_key_by_val}.csv", index=False)
else:
    print("Repeated CV skipped because the selected model is not stored.")

## 6.5 Nested Cross-Validation

Nested CV gives a less biased estimate of generalization because model tuning happens inside the inner loop, and performance is measured on unseen outer folds.  
This cell is computationally heavier, so it uses a smaller tuning budget in `QUICK_RUN` mode.

In [ ]:
def nested_cv_for_model(
    estimator: Any,
    param_distributions: Dict[str, List[Any]],
    model_label: str,
    scale: bool = False,
    outer_splits: int = 5,
    inner_splits: int = 4,
    n_iter: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Run group-safe nested CV for one selected model family on the full development set."""
    # Development set = train + validation. Locked test remains excluded.
    X_dev = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_dev = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
    groups_dev = pd.concat([groups_train, groups_val], axis=0).reset_index(drop=True)

    bins_dev = make_target_bins(y_dev)
    try:
        outer_cv = StratifiedGroupKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
        outer_iterator = list(outer_cv.split(np.zeros(len(y_dev)), bins_dev, groups_dev))
        outer_name = "StratifiedGroupKFold"
    except Exception as exc:
        print(f"Outer StratifiedGroupKFold failed; using GroupKFold. Reason: {type(exc).__name__}: {exc}")
        outer_cv = GroupKFold(n_splits=outer_splits)
        outer_iterator = list(outer_cv.split(np.zeros(len(y_dev)), y_dev, groups_dev))
        outer_name = "GroupKFold"

    outer_rows = []
    for outer_fold, (tr, te) in enumerate(outer_iterator, start=1):
        X_inner_train = X_dev.iloc[tr].reset_index(drop=True)
        y_inner_train = y_dev.iloc[tr].reset_index(drop=True)
        g_inner_train = groups_dev.iloc[tr].reset_index(drop=True)
        X_outer_test = X_dev.iloc[te].reset_index(drop=True)
        y_outer_test = y_dev.iloc[te].reset_index(drop=True)

        inner_cv_splits, inner_name = make_train_cv_splits(y_inner_train, g_inner_train, n_splits=inner_splits, seed=RANDOM_STATE + outer_fold)
        pipe = build_pipeline(clone(estimator), scale=scale)
        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_distributions,
            n_iter=min(n_iter, max(1, math.prod([len(v) for v in param_distributions.values()]))),
            scoring="r2",
            cv=inner_cv_splits,
            random_state=RANDOM_STATE + outer_fold,
            n_jobs=N_JOBS,
            error_score=np.nan,
        )
        search.fit(X_inner_train, y_inner_train)
        pred_outer = search.best_estimator_.predict(X_outer_test)
        m = regression_metrics(y_outer_test, pred_outer, prefix="")
        outer_rows.append({
            "Model": model_label,
            "Outer_Fold": outer_fold,
            "Outer_CV": outer_name,
            "Inner_CV": inner_name,
            "Best_Inner_CV_R2": float(search.best_score_),
            **m,
        })

    outer_table = pd.DataFrame(outer_rows)
    summary = outer_table[["R2", "RMSE", "MAE"]].agg(["mean", "std"]).T.reset_index()
    summary.columns = ["Metric", "Mean", "SD"]
    summary.insert(0, "Model", model_label)
    return outer_table, summary

RUN_NESTED_CV = False if QUICK_RUN else True

if RUN_NESTED_CV:
    # Example: nested CV for XGBoost if available; otherwise Random Forest.
    if HAS_XGBOOST:
        nested_estimator = XGBRegressor(objective="reg:squarederror", eval_metric="rmse", random_state=RANDOM_STATE, n_jobs=N_JOBS, tree_method="hist")
        nested_params = xgb_params
        nested_label = "XGBoost_NestedCV"
        nested_scale = False
    else:
        nested_estimator = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS)
        nested_params = rf_params
        nested_label = "RandomForest_NestedCV"
        nested_scale = False

    nested_table, nested_summary = nested_cv_for_model(
        nested_estimator,
        nested_params,
        nested_label,
        scale=nested_scale,
        outer_splits=5,
        inner_splits=4,
        n_iter=15 if QUICK_RUN else 30,
    )
    display(nested_summary)
    display(nested_table)
    nested_table.to_csv(TABLE_DIR / "nested_cv_outer_folds.csv", index=False)
    nested_summary.to_csv(TABLE_DIR / "nested_cv_summary.csv", index=False)
else:
    print("Nested CV skipped in QUICK_RUN mode. Set RUN_NESTED_CV = True or QUICK_RUN = False for the paper run.")

## 6.6 Locked-Test Evaluation — Final Stage Only

This is the only cell allowed to score the locked test.  
Run it only after the model family, features, tuning strategy, and selection rule are finalized.

In [ ]:
def select_best_model_key(results: pd.DataFrame) -> str:
    """Select the best model using validation R2 only, not the locked test."""
    sorted_results = results.sort_values(["Validation_R2", "Train_Validation_R2_Gap"], ascending=[False, True])
    return str(sorted_results.iloc[0]["Stored_Model_Key"])

best_final_key = select_best_model_key(results_df)
selection_model = FITTED_MODELS.get(best_final_key)

print(f"Selected model by validation only: {best_final_key}")

locked_test_row = None
final_model = None

if REVEAL_LOCKED_TEST:
    if selection_model is None:
        raise RuntimeError("Selected model is not a stored sklearn Pipeline. Use a stored model for final locked-test evaluation.")

    # Refit on the full development set: train + validation.
    X_dev = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_dev = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

    final_model = clone(selection_model)
    final_model.fit(X_dev, y_dev)

    pred_train_selection = selection_model.predict(X_train)
    pred_val_selection = selection_model.predict(X_val)
    pred_locked_test = final_model.predict(X_locked_test)

    locked_test_row = {
        "Final_Model_Key": best_final_key,
        **regression_metrics(y_train, pred_train_selection, prefix="Train_Selection_"),
        **regression_metrics(y_val, pred_val_selection, prefix="Validation_Selection_"),
        **regression_metrics(y_locked_test, pred_locked_test, prefix="LockedTest_Final_"),
    }
    locked_test_row["Train_Validation_R2_Gap"] = locked_test_row["Train_Selection_R2"] - locked_test_row["Validation_Selection_R2"]
    locked_test_row["Train_Test_R2_Gap"] = locked_test_row["Train_Selection_R2"] - locked_test_row["LockedTest_Final_R2"]

    display(pd.DataFrame([locked_test_row]))
    joblib.dump(final_model, MODEL_DIR / f"FINAL_{best_final_key}.joblib")
    pd.DataFrame([locked_test_row]).to_csv(TABLE_DIR / "locked_test_final_evaluation.csv", index=False)
    print("Locked test has now been used once. Do not keep re-selecting models after seeing this result.")
else:
    print("Locked test remains hidden. Set REVEAL_LOCKED_TEST = True only for the final one-time evaluation.")
    print("The selected model has not been refit on train+validation yet because test scoring is protected.")

# Step 7: Diagnostics and Visualization

This section creates publication-style diagnostic tables and figures.  
When the locked test is hidden, diagnostics are made for training and validation only. After final reveal, locked-test plots can also be generated.

## 7.1 Model Performance Table

This cell builds a consolidated performance table. OOF, repeated CV, nested CV, and locked-test fields are added when those cells have been run.

In [ ]:
performance_table = results_df.copy()

# Add OOF summary for the best model if available.
try:
    performance_table.loc[performance_table["Stored_Model_Key"] == best_key_by_val, "OOF_R2"] = oof_metric["OOF_R2"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_key_by_val, "OOF_RMSE"] = oof_metric["OOF_RMSE"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_key_by_val, "OOF_MAE"] = oof_metric["OOF_MAE"]
except Exception:
    pass

try:
    rep_r2 = repeated_summary.loc[repeated_summary["Metric"] == "R2", ["Mean", "SD"]].iloc[0]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_key_by_val, "RepeatedCV_R2_Mean"] = rep_r2["Mean"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_key_by_val, "RepeatedCV_R2_SD"] = rep_r2["SD"]
except Exception:
    pass

if locked_test_row is not None:
    performance_table.loc[performance_table["Stored_Model_Key"] == best_final_key, "LockedTest_R2"] = locked_test_row["LockedTest_Final_R2"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_final_key, "LockedTest_RMSE"] = locked_test_row["LockedTest_Final_RMSE"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_final_key, "LockedTest_MAE"] = locked_test_row["LockedTest_Final_MAE"]
    performance_table.loc[performance_table["Stored_Model_Key"] == best_final_key, "Train_Test_R2_Gap"] = locked_test_row["Train_Test_R2_Gap"]

cols_first = [
    "Stored_Model_Key", "Model", "Train_R2", "OOF_R2", "Validation_R2", "RepeatedCV_R2_Mean",
    "RepeatedCV_R2_SD", "LockedTest_R2", "Train_RMSE", "Validation_RMSE", "LockedTest_RMSE",
    "Train_MAE", "Validation_MAE", "LockedTest_MAE", "Train_Validation_R2_Gap", "Train_Test_R2_Gap", "Overfit_Risk",
]
ordered_cols = [c for c in cols_first if c in performance_table.columns] + [c for c in performance_table.columns if c not in cols_first]
performance_table = performance_table[ordered_cols]

display(performance_table.drop(columns=["Best_Params"], errors="ignore"))
performance_table.to_csv(TABLE_DIR / "model_performance_full_table.csv", index=False)

## 7.2 Measured vs Predicted Plot

These plots compare measured rut depth against predicted rut depth. The 1:1 line represents perfect prediction.

In [ ]:
def prediction_frame(model: Pipeline, X_data: pd.DataFrame, y_true: pd.Series, split_name: str) -> pd.DataFrame:
    """Create a measured/predicted dataframe for diagnostics."""
    pred = model.predict(X_data)
    out = pd.DataFrame({"Split": split_name, "Measured": y_true.values, "Predicted": pred})
    out["Residual"] = out["Predicted"] - out["Measured"]
    out["Abs_Error"] = out["Residual"].abs()
    out["Squared_Error"] = out["Residual"] ** 2
    denom = out["Measured"].abs().replace(0, np.nan)
    out["Abs_Relative_Error_pct"] = 100.0 * out["Abs_Error"] / denom
    return out

if selection_model is None:
    selection_model = FITTED_MODELS[best_key_by_val]

pred_train_df = prediction_frame(selection_model, X_train, y_train, "Train")
pred_val_df = prediction_frame(selection_model, X_val, y_val, "Validation")
pred_frames = [pred_train_df, pred_val_df]

if locked_test_row is not None and final_model is not None:
    pred_test_df = prediction_frame(final_model, X_locked_test, y_locked_test, "Locked_Test")
    pred_frames.append(pred_test_df)

all_predictions_df = pd.concat(pred_frames, ignore_index=True)
all_predictions_df.to_csv(TABLE_DIR / "measured_vs_predicted_predictions.csv", index=False)

for split_name, sub in all_predictions_df.groupby("Split"):
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(sub["Measured"], sub["Predicted"], alpha=0.7)
    mn = min(sub["Measured"].min(), sub["Predicted"].min())
    mx = max(sub["Measured"].max(), sub["Predicted"].max())
    ax.plot([mn, mx], [mn, mx], linestyle="--")
    ax.set_title(f"Measured vs Predicted — {split_name}")
    ax.set_xlabel("Measured Rut_20k (mm)")
    ax.set_ylabel("Predicted Rut_20k (mm)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"measured_vs_predicted_{split_name}.png", dpi=200)
    plt.show()

## 7.3 Residual Plots

Residual plots show bias patterns. A good regression model should not show strong systematic residual trends across predicted values or important variables.

In [ ]:
for split_name, sub in all_predictions_df.groupby("Split"):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(sub["Predicted"], sub["Residual"], alpha=0.7)
    ax.axhline(0, linestyle="--")
    ax.set_title(f"Residuals vs Predicted — {split_name}")
    ax.set_xlabel("Predicted Rut_20k (mm)")
    ax.set_ylabel("Residual = Predicted - Measured")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"residuals_vs_predicted_{split_name}.png", dpi=200)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(sub["Residual"], bins=25, edgecolor="black")
    ax.set_title(f"Residual Distribution — {split_name}")
    ax.set_xlabel("Residual")
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"residual_distribution_{split_name}.png", dpi=200)
    plt.show()

# Residuals against important variables for validation data.
important_residual_vars = [c for c in ["PG_HighTemp", "RBR_JMF_fraction", "Va", "VMA", "VFA", "Dust_Binder", "AsphaltContent_Design"] if c in X_val.columns]
val_diag = X_val.copy()
val_diag["Residual"] = pred_val_df["Residual"].values

for col in important_residual_vars:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(val_diag[col], val_diag["Residual"], alpha=0.7)
    ax.axhline(0, linestyle="--")
    ax.set_title(f"Validation residuals vs {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Residual = Predicted - Measured")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"validation_residuals_vs_{re.sub(r'[^A-Za-z0-9]+', '_', col)}.png", dpi=200)
    plt.show()

## 7.4 Error by Rutting Range

This cell divides rutting into low, medium, high, and very high ranges.  
This helps identify whether the model underpredicts high rutting or overpredicts low rutting.

In [ ]:
def rut_range_label(value: float) -> str:
    """Engineering ranges for rut depth interpretation."""
    if value < 2:
        return "Low <2 mm"
    if value < 5:
        return "Medium 2-5 mm"
    if value < 7:
        return "High 5-7 mm"
    return "Very high >7 mm"


def error_by_group(pred_df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Calculate error metrics by an engineering grouping column."""
    rows = []
    for (split, group), sub in pred_df.groupby(["Split", group_col]):
        if len(sub) < 2:
            continue
        rows.append({
            "Split": split,
            group_col: group,
            "Rows": len(sub),
            **regression_metrics(sub["Measured"], sub["Predicted"], prefix=""),
            "Mean_Bias_PredMinusMeas": float(sub["Residual"].mean()),
        })
    return pd.DataFrame(rows)

all_predictions_df["Rut_Range"] = all_predictions_df["Measured"].apply(rut_range_label)
rut_range_error = error_by_group(all_predictions_df, "Rut_Range")
display(rut_range_error)
rut_range_error.to_csv(TABLE_DIR / "error_by_rutting_range.csv", index=False)

print("Interpretation: high MAE or negative bias in 'Very high >7 mm' suggests high-rut compression/underprediction.")

## 7.5 Error by RBR Band

This cell groups errors by RBR band. It helps determine whether the model is reliable for low-RBR and high-RBR mixtures.

In [ ]:
def attach_rbr_band(pred_df: pd.DataFrame, split_name: str, source_df: pd.DataFrame) -> pd.DataFrame:
    """Attach RBR_band from the original dataframe for the requested split."""
    out = pred_df.copy()
    if "RBR_band" not in df.columns:
        out["RBR_band"] = "RBR_band unavailable"
        return out
    if split_name == "Train":
        out["RBR_band"] = df.iloc[train_idx]["RBR_band"].reset_index(drop=True).astype(str).values
    elif split_name == "Validation":
        out["RBR_band"] = df.iloc[val_idx]["RBR_band"].reset_index(drop=True).astype(str).values
    elif split_name == "Locked_Test":
        out["RBR_band"] = df.iloc[test_idx]["RBR_band"].reset_index(drop=True).astype(str).values
    return out

rbr_frames = []
for split_name, sub in all_predictions_df.groupby("Split"):
    rbr_frames.append(attach_rbr_band(sub.reset_index(drop=True), split_name, df))
all_predictions_rbr_df = pd.concat(rbr_frames, ignore_index=True)

rbr_error = error_by_group(all_predictions_rbr_df, "RBR_band")
display(rbr_error)
rbr_error.to_csv(TABLE_DIR / "error_by_rbr_band.csv", index=False)

print("Interpretation: if high-RBR mixtures have higher MAE or bias, the model may need more high-RBR data or binder/rheology variables.")

## 7.6 SHAP Explainability

SHAP helps explain feature influence. The plot is statistical, so the written interpretation must connect the feature effects to asphalt engineering behavior.

In [ ]:
def get_transformed_feature_names(pipe: Pipeline) -> List[str]:
    """Return feature names after ColumnTransformer/OneHotEncoder."""
    preprocess = pipe.named_steps["preprocess"]
    try:
        return list(preprocess.get_feature_names_out())
    except Exception:
        return [f"x{i}" for i in range(preprocess.transform(X_train.iloc[:1]).shape[1])]

RUN_SHAP = HAS_SHAP

if RUN_SHAP and selection_model is not None:
    try:
        model_step = selection_model.named_steps["model"]
        preprocess_step = selection_model.named_steps["preprocess"]
        X_val_trans = preprocess_step.transform(X_val)
        feature_names_trans = get_transformed_feature_names(selection_model)

        # Limit rows for speed.
        max_rows = min(500, X_val_trans.shape[0])
        X_shap = X_val_trans[:max_rows]

        explainer = shap.Explainer(model_step, X_shap)
        shap_values = explainer(X_shap)

        shap.summary_plot(shap_values, features=X_shap, feature_names=feature_names_trans, show=False)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "shap_summary_beeswarm.png", dpi=200, bbox_inches="tight")
        plt.show()

        mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
        shap_importance = pd.DataFrame({"Feature": feature_names_trans, "MeanAbsSHAP": mean_abs_shap}).sort_values("MeanAbsSHAP", ascending=False)
        display(shap_importance.head(20))
        shap_importance.to_csv(TABLE_DIR / "shap_importance.csv", index=False)

        shap.summary_plot(shap_values, features=X_shap, feature_names=feature_names_trans, plot_type="bar", show=False)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "shap_summary_bar.png", dpi=200, bbox_inches="tight")
        plt.show()
    except Exception as exc:
        print(f"SHAP could not run for this selected model: {type(exc).__name__}: {exc}")
else:
    print("SHAP skipped because shap is unavailable or no sklearn pipeline was selected.")

## 7.7 Partial Dependence Plots

PDPs show average model response as one or two features change. Use them to check whether trends make engineering sense.

In [ ]:
RUN_PDP = True
pdp_features = [c for c in ["PG_HighTemp", "RBR_JMF_fraction", "Va", "VMA", "VFA", "Dust_Binder", "AsphaltContent_Design"] if c in X_train.columns]

if RUN_PDP and selection_model is not None and pdp_features:
    for feat in pdp_features[:6]:
        try:
            fig, ax = plt.subplots(figsize=(6, 4.5))
            PartialDependenceDisplay.from_estimator(selection_model, X_train, [feat], ax=ax)
            ax.set_title(f"1D PDP — {feat}")
            plt.tight_layout()
            plt.savefig(FIG_DIR / f"pdp_1d_{re.sub(r'[^A-Za-z0-9]+', '_', feat)}.png", dpi=200)
            plt.show()
        except Exception as exc:
            print(f"PDP failed for {feat}: {type(exc).__name__}: {exc}")

    # Useful 2D interactions when both variables exist.
    pdp_pairs = [("PG_HighTemp", "RBR_JMF_fraction"), ("Va", "VFA"), ("Dust_Binder", "RBR_JMF_fraction")]
    for a, b in pdp_pairs:
        if a in X_train.columns and b in X_train.columns:
            try:
                fig, ax = plt.subplots(figsize=(6, 5))
                PartialDependenceDisplay.from_estimator(selection_model, X_train, [(a, b)], ax=ax)
                ax.set_title(f"2D PDP — {a} × {b}")
                plt.tight_layout()
                plt.savefig(FIG_DIR / f"pdp_2d_{re.sub(r'[^A-Za-z0-9]+', '_', a)}_{re.sub(r'[^A-Za-z0-9]+', '_', b)}.png", dpi=200)
                plt.show()
            except Exception as exc:
                print(f"2D PDP failed for {a}, {b}: {type(exc).__name__}: {exc}")
else:
    print("PDP skipped because no selected sklearn pipeline or no PDP features were available.")

## 7.8 Applicability-Domain Analysis

Applicability domain checks whether validation/test samples fall outside the training feature space.  
Predictions outside the training domain should be interpreted with caution.

In [ ]:
def applicability_domain_numeric(X_fit: pd.DataFrame, X_check: pd.DataFrame, check_name: str, numeric_cols: List[str], threshold_quantile: float = 0.95) -> pd.DataFrame:
    """Nearest-neighbor applicability domain using standardized numeric features.

    Training threshold is based on each training row's distance to its nearest other training row.
    Check rows farther than the threshold are flagged as outside the training domain.
    """
    if len(numeric_cols) < 2:
        return pd.DataFrame()

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_fit_num = prep.fit_transform(X_fit[numeric_cols])
    X_check_num = prep.transform(X_check[numeric_cols])

    nn_train = NearestNeighbors(n_neighbors=min(2, len(X_fit_num))).fit(X_fit_num)
    train_distances, _ = nn_train.kneighbors(X_fit_num)
    # Distance to the nearest other training point; if only one row, use zero.
    train_nearest_other = train_distances[:, 1] if train_distances.shape[1] > 1 else train_distances[:, 0]
    threshold = np.quantile(train_nearest_other, threshold_quantile)

    nn = NearestNeighbors(n_neighbors=1).fit(X_fit_num)
    check_distances, _ = nn.kneighbors(X_check_num)
    check_distances = check_distances.ravel()

    return pd.DataFrame({
        "Split": check_name,
        "Nearest_Train_Distance": check_distances,
        "AD_Threshold": threshold,
        "Outside_Applicability_Domain": check_distances > threshold,
    })

ad_train = applicability_domain_numeric(X_train, X_train, "Train", numeric_features)
ad_val = applicability_domain_numeric(X_train, X_val, "Validation", numeric_features)
ad_tables = [ad_train, ad_val]
if locked_test_row is not None:
    ad_test = applicability_domain_numeric(X_train, X_locked_test, "Locked_Test", numeric_features)
    ad_tables.append(ad_test)

applicability_df = pd.concat([t for t in ad_tables if not t.empty], ignore_index=True) if ad_tables else pd.DataFrame()

if not applicability_df.empty:
    display(applicability_df.groupby("Split")["Outside_Applicability_Domain"].agg(["count", "sum", "mean"]).reset_index())
    applicability_df.to_csv(TABLE_DIR / "applicability_domain_flags.csv", index=False)

    # Join validation AD flags to validation errors.
    val_ad_error = pred_val_df.copy()
    val_ad_error["Outside_Applicability_Domain"] = ad_val["Outside_Applicability_Domain"].values
    ad_error_summary = val_ad_error.groupby("Outside_Applicability_Domain").apply(
        lambda s: pd.Series({
            "Rows": len(s),
            "R2": r2_score(s["Measured"], s["Predicted"]) if len(s) > 1 else np.nan,
            "RMSE": rmse(s["Measured"], s["Predicted"]) if len(s) > 1 else np.nan,
            "MAE": mean_absolute_error(s["Measured"], s["Predicted"]) if len(s) > 1 else np.nan,
        })
    ).reset_index()
    display(ad_error_summary)
    ad_error_summary.to_csv(TABLE_DIR / "applicability_domain_validation_error_summary.csv", index=False)
else:
    print("Applicability domain skipped because fewer than two numeric features were available.")

# Step 8: Engineering Interpretation

This final section creates a report-style interpretation that connects the statistical results to asphalt mixture behavior.

## 8.1–8.5 Final Report-Style Interpretation

This cell writes an engineering summary file using the available results. Edit the text after reviewing the actual figures and tables.

In [ ]:
def safe_best_value(table: pd.DataFrame, col: str, default: str = "not available") -> str:
    """Return a formatted best-model value if the column exists."""
    if col not in table.columns or table.empty or pd.isna(table.iloc[0].get(col, np.nan)):
        return default
    value = table.iloc[0][col]
    if isinstance(value, (float, np.floating)):
        return f"{value:.3f}"
    return str(value)

best_row = performance_table.sort_values("Validation_R2", ascending=False).iloc[0]
best_model_name = best_row.get("Stored_Model_Key", "Selected model")
train_r2_txt = safe_best_value(performance_table.sort_values("Validation_R2", ascending=False), "Train_R2")
val_r2_txt = safe_best_value(performance_table.sort_values("Validation_R2", ascending=False), "Validation_R2")
locked_r2_txt = safe_best_value(performance_table.sort_values("Validation_R2", ascending=False), "LockedTest_R2")
gap_txt = safe_best_value(performance_table.sort_values("Validation_R2", ascending=False), "Train_Validation_R2_Gap")

try:
    top_features_txt = ", ".join(shap_importance["Feature"].head(8).astype(str).tolist())
except Exception:
    # Fall back to tree feature_importances_ when SHAP is unavailable.
    try:
        model_step = selection_model.named_steps["model"]
        names = get_transformed_feature_names(selection_model)
        imp = pd.Series(model_step.feature_importances_, index=names).sort_values(ascending=False)
        top_features_txt = ", ".join(imp.head(8).index.astype(str).tolist())
    except Exception:
        top_features_txt = "feature importance not available"

engineering_report = f"""
# Engineering Interpretation Summary — Rutting ML Model

## 8.1 Whether the model learns real mixture behavior

The selected model is **{best_model_name}**. The training R² is **{train_r2_txt}**, the validation R² is **{val_r2_txt}**, and the train-validation R² gap is **{gap_txt}**. The most influential variables identified by SHAP or model importance are: **{top_features_txt}**.

These variables should be interpreted using asphalt mixture behavior, not only statistical ranking. Binder grade/high-temperature stiffness, true RBR, RAP binder contribution, air voids, VMA/VFA, dust-to-binder ratio, gradation passing values, NMAS, aggregate angularity, and absorption can all influence rutting resistance because they affect mixture stiffness, aggregate skeleton stability, binder availability, and compaction structure.

## 8.2 Model limitations

The current model relies mainly on routine JMF/volumetric/gradation variables. Its prediction ceiling may be limited because important physical variables are missing or simplified, including continuous binder PG details, binder rheology, aging condition, test temperature, loading condition, aggregate source/type, compaction effort, plant production variables, and specimen preparation details.

## 8.3 Reliability of predictions

Use the validation error by rutting range and RBR band tables to identify where predictions are strongest and weakest. If the high-rutting range has higher MAE or negative bias, the model is compressing the upper tail and may underpredict severe rutting. If high-RBR mixtures have larger error, the training data may not fully represent recycled-binder behavior.

Locked-test R² is **{locked_r2_txt}**. If this value is not available, the locked test has not been revealed yet, which is correct during model selection.

## 8.4 Outlier and data-quality interpretation

The IQR method flags univariate extremes, while Isolation Forest flags unusual multivariable combinations. Outliers should not be removed automatically. Keep outliers that represent valid extreme engineering cases, investigate values that violate physical/specification logic, and correct or exclude records only when source verification confirms a data-entry, unit, merge, or lab-processing error.

## 8.5 Final modeling recommendation

For publication, report the full validation ladder: training, OOF, validation, repeated CV, nested CV, and one-time locked test. The best model should be selected using validation and robustness metrics only. The locked test should support the final engineering conclusion but should not be used to choose or tune the model.

The model is publication-ready only if the final report includes: leakage-safe group splitting by MixDesignKey, full preprocessing pipelines, honest validation, residual/error diagnostics, SHAP/PDP interpretation, applicability-domain limits, and a clear explanation of data limitations and missing physical inputs.
""".strip()

report_path = OUTPUT_DIR / "Engineering_Interpretation_Summary.md"
report_path.write_text(engineering_report, encoding="utf-8")
print(engineering_report)
print(f"\nSaved report summary to: {report_path}")

# Final Save Cell

This cell saves the main outputs created by the notebook.

In [ ]:
# Save fitted sklearn models from the development workflow.
for key, model in FITTED_MODELS.items():
    safe_key = re.sub(r"[^A-Za-z0-9_.-]+", "_", key)
    joblib.dump(model, MODEL_DIR / f"{safe_key}.joblib")

# Save CV search tables.
for key, table in CV_RESULT_TABLES.items():
    safe_key = re.sub(r"[^A-Za-z0-9_.-]+", "_", key)
    table.to_csv(TABLE_DIR / f"cv_results_{safe_key}.csv", index=False)

print("Saved outputs:")
print("- Figures:", FIG_DIR)
print("- Tables:", TABLE_DIR)
print("- Models:", MODEL_DIR)
print("- Splits:", SPLIT_DIR)
print("Reminder: Do not reveal the locked test until the final evaluation stage.")